In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
pip install simfin

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 677.9 kB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:

# Installation: pip install simfin
# import simfin package
import simfin as sf

# Set your API-key for downloading data.
# Key is read from the environment / .env (never hardcode it).
import os
from dotenv import load_dotenv
load_dotenv()
sf.set_api_key(os.environ['SIMFIN_API_KEY'])

# Set the local directory where data-files are stored.
# The directory will be created if it does not already exist.
sf.set_data_dir('~/simfin_data/')

# Download the data from the SimFin server and load into a Pandas DataFrame.
df5 = sf.load_shareprices(variant='daily')

# Print the first rows of the data.
print(df5.head())

#print(df10.head())
df11=sf.load_derived(variant='quarterly')


#print(df10.head())
#df11=sf.load_derived(variant='quarterly')

print(df11)

# Download the data from the SimFin server and load into a Pandas DataFrame.
df_ind = sf.load_industries()

# Print the first rows of the data.
print(df_ind.head())


Dataset "us-shareprices-daily" not on disk.
- Downloading ... 100.0%
- Extracting zip-file ... Done!
- Loading from disk ... Done!
                   SimFinId   Open   High    Low  Close  Adj. Close   Volume  \
Ticker Date                                                                    
A      2000-01-03     45846  78.75  78.94  67.38  72.00       45.85  3343600   
       2000-01-04     45846  68.13  68.88  64.75  66.50       42.35  3408500   
       2000-01-05     45846  66.25  66.31  61.31  61.56       39.20  4119200   
       2000-01-06     45846  61.63  62.00  58.13  60.00       38.21  1812900   
       2000-01-07     45846  59.06  65.94  59.00  65.00       41.39  2016900   

                   Dividend  Shares Outstanding  
Ticker Date                                      
A      2000-01-03       NaN                 NaN  
       2000-01-04       NaN                 NaN  
       2000-01-05       NaN                 NaN  
       2000-01-06       NaN                 NaN  
       2

In [4]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)


df6=df5

df6['dt'] = df6.index.get_level_values('Date') 
df6['tk'] = df6.index.get_level_values('Ticker') 





df6['yr'] = pd.DatetimeIndex(df6['dt']).year
df6['mth'] = pd.DatetimeIndex(df6['dt']).month
df6['dd'] = pd.DatetimeIndex(df6['dt']).day

df6.tail()


#df6.dtypes

#df_yr1_3.describe()



SimFinId  Open  High   Low  Close  Adj. Close  Volume  \
Ticker Date                                                                
NaN    2026-10-06  18728220  2.19  2.22  2.13   2.17        2.17   24963   
       2026-10-07  18728220  2.19  2.21  2.11   2.18        2.18   26546   
       2026-10-08  18728220  2.13  2.21  2.11   2.14        2.14   20370   
       2026-10-09  18728220  2.19  2.19  2.10   2.11        2.11   28335   
       2026-10-12  18728220  2.11  2.16  1.92   1.92        1.92   30834   

                   Dividend  Shares Outstanding         dt   tk    yr  mth  dd  
Ticker Date                                                                     
NaN    2026-10-06       NaN          23496833.0 2026-10-06  NaN  2026   10   6  
       2026-10-07       NaN          23496833.0 2026-10-07  NaN  2026   10   7  
       2026-10-08       NaN          23496833.0 2026-10-08  NaN  2026   10   8  
       2026-10-09       NaN          23496833.0 2026-10-09  NaN  2026   10   9  
       2026-10-12       NaN          23496833.0 2026-10-12  NaN  2026   10  12

In [5]:
def get_close_price(df,opendt, cal_days):
    
    dftmp = df.copy()
    dftmp['open_dt']=opendt
    dftmp['days_from_open']=pd.to_datetime(dftmp['dt'], format='%Y-%m-%d')-pd.to_datetime(dftmp['open_dt'], format='%Y-%m-%d') 
    
    dftmp['cal_days']=cal_days
    temp1 = dftmp['cal_days'].apply(np.ceil).apply(lambda x: pd.Timedelta(x, unit='D'))
    
    dftmp1=dftmp[dftmp.days_from_open<temp1]


    dftmp1_close_dt=dftmp1.groupby(['tk']).dt.agg([max])

    dftmp1_close_dt['close_dt']=dftmp1_close_dt['max']
    df_tmp2 = pd.merge(dftmp1, dftmp1_close_dt, how="inner", on=["tk"])
    df_tmp2_close=df_tmp2[df_tmp2.close_dt == df_tmp2.dt]
    
    retds=df_tmp2_close[["tk", "close_dt", "Close"]]
    return retds

In [6]:
import datetime
from datetime import timedelta 
from datetime import datetime
#from datetime import strptime
def get_current_state(date1, date2,date0, p_by_o_high,p_by_o_low):
    
    #df_yr1=df6.loc[ (df6.dt >= date1.strftime("%Y-%m-%d")) & (df6.dt <=  date2.strftime("%Y-%m-%d"))]
    #print("df_yr1.head=",df_yr1.head()
    df_yr1=df6.loc[ (df6.dt >= date1) & (df6.dt <=  date2)]

    df_yr1['calc_price']=0.5*(df_yr1['Low']+df_yr1['High'])
    x=0
    d=date1
    dcp=df_yr1.copy()
    while x<1: 
        
        df_yr1_open1=dcp[dcp.dt == d]
        #print("date1 d=",d, df_yr1_open1.empty)
        if df_yr1_open1.shape[0]<400:
            #t=datetime.datetime.strptime(d, "%Y-%m-%d")+timedelta(days=1)
            #t=datetime.strptime(d, "%Y-%m-%d")+timedelta(days=1)
            t=datetime.datetime.strptime(d, "%Y-%m-%d")+timedelta(days=1)
            print("t 1=",t)
            d=t.strftime("%Y-%m-%d")
            
            print("d 1=",d)
            dcp=df_yr1.copy()
        else:
            x=1
           
   
    print("actual date1=",d)
    df_yr1_open1['calc_open']=df_yr1_open1['calc_price']
    df_yr1_open=df_yr1_open1[['tk','calc_open','Volume']]
    print("df_yr1_open=",df_yr1_open)
    
    #df_yr1_close1=df_yr1[df_yr1.dt == date2]
 
    x=0
    d=date2
    dcp=df_yr1.copy()
    while x<1:   
        
        df_yr1_close1=dcp[dcp.dt == d]
        #print("date2 d=",d, df_yr1_close1.empty)
        if  df_yr1_close1.shape[0]<400:
            #t=datetime.strptime(d, '%m/%d/%y')+timedelta(days=1)
            
            #t=datetime.datetime.strptime(d, "%Y-%m-%d")-timedelta(days=1)
            #t=datetime.strptime(d, "%Y-%m-%d")-timedelta(days=1)
            t=datetime.datetime.strptime(d, "%Y-%m-%d")-timedelta(days=1)
            print("t 2=",t)
            d=t.strftime("%Y-%m-%d")
           
            print("d 2=",d)
            dcp=df_yr1.copy()
        else:
            x=1
    print("actual date2=",d)       
            
    df_yr1_close1['calc_close']=df_yr1_close1['calc_price']
    df_yr1_close=df_yr1_close1[['tk','dt','calc_close']]
    print("df_yr1_close=",df_yr1_close)
    
    ds_with_open = pd.merge(df_yr1, df_yr1_open, how="inner", on=["tk"])
    
  
    
    #ds_with_max['max_reached']=(ds_with_max['High'] == ds_with_max['year_high'])
    
    #ds_with_open['w1']=(ds_with_open['calc_price']>=ds_with_open['calc_open']*p_by_o_high)
   # ds_with_open['w2']=(ds_with_open['calc_price']<=ds_with_open['calc_open']*p_by_o_low)

    ds_with_open.loc[ds_with_open['calc_price'] >=ds_with_open['calc_open']*p_by_o_high, 'w1'] = 1 
    ds_with_open.loc[ds_with_open['calc_price'] <ds_with_open['calc_open']*p_by_o_high, 'w1'] = 0 
    
    ds_with_open.loc[ds_with_open['calc_price'] <=ds_with_open['calc_open']*p_by_o_low, 'w2'] = 1 
    ds_with_open.loc[ds_with_open['calc_price'] >ds_with_open['calc_open']*p_by_o_low, 'w2'] = 0 
    
    gp_ds=ds_with_open[ds_with_open.w1==1]
    dp_ds=ds_with_open[ds_with_open.w2==1]
    
    gp_sum=gp_ds.groupby(['tk']).dt.agg([min])
    gp_sum['min_gp_dt']=gp_sum['min']
    
    dp_sum=dp_ds.groupby(['tk']).dt.agg([min])
    dp_sum['min_dp_dt']=dp_sum['min']
    
    #print("gp_sum=",gp_sum)
    
    #print("dp_sum=",dp_sum)
    
    #w1ds=ds_with_open[(ds_with_open.w1==1)]
    
    #print("w1ds=",w1ds)
    
    
    df_max_w1=ds_with_open.groupby(['tk']).w1.agg([max])
    df_max_w1['max_w1']=df_max_w1['max']
    #print(df_max_dt.head())
    
    df_max_w2=ds_with_open.groupby(['tk']).w2.agg([max])
    df_max_w2['max_w2']=df_max_w2['max']
    
    ds_maxw1_maxw2 = pd.merge(df_max_w1, df_max_w2, how="inner", on=["tk"])
    ds_maxw1_maxw2_gp_sum = pd.merge(ds_maxw1_maxw2, gp_sum, how="left", on=["tk"])
    ds_maxw1_maxw2_gp_sum_dp_sum = pd.merge(ds_maxw1_maxw2_gp_sum, dp_sum, how="left", on=["tk"])
    
    ds_maxw1_maxw2_gp_sum_dp_sum.loc[ds_maxw1_maxw2_gp_sum_dp_sum['min_gp_dt'].isnull(), 'min_gp_dt'] = date2 
    ds_maxw1_maxw2_gp_sum_dp_sum.loc[ds_maxw1_maxw2_gp_sum_dp_sum['min_dp_dt'].isnull() , 'min_dp_dt'] = date1 
    
    ds_maxw1_maxw2_gp_sum_dp_sum.loc[ds_maxw1_maxw2_gp_sum_dp_sum['min_gp_dt'] <ds_maxw1_maxw2_gp_sum_dp_sum['min_dp_dt'] , 'w3'] = 1 
    ds_maxw1_maxw2_gp_sum_dp_sum.loc[ds_maxw1_maxw2_gp_sum_dp_sum['min_gp_dt'] >=ds_maxw1_maxw2_gp_sum_dp_sum['min_dp_dt'] , 'w3'] = 0 
    
    #ds_maxw1_maxw2_gp_sum_dp_sum.loc[ds_maxw1_maxw2_gp_sum_dp_sum['max_w1']*max(1-ds_maxw1_maxw2_gp_sum_dp_sum['max_w2'],ds_maxw1_maxw2_gp_sum_dp_sum['w3']) ==1 , 'y'] = 1 
    #ds_maxw1_maxw2_gp_sum_dp_sum.loc[ds_maxw1_maxw2_gp_sum_dp_sum['max_w1']*max(1-ds_maxw1_maxw2_gp_sum_dp_sum['max_w2'],ds_maxw1_maxw2_gp_sum_dp_sum['w3']) ==0 , 'y'] = 0
    
    ds_sucess1 = pd.merge(df_yr1_open, ds_maxw1_maxw2_gp_sum_dp_sum, how="inner", on=["tk"])
    #print("ds_sucess1=",ds_sucess1)
    ds_sucess = pd.merge(ds_sucess1, df_yr1_close, how="inner", on=["tk"])
    
    ds_sucess['y']=ds_sucess['max_w1'] * (1-ds_sucess['max_w2']*(1-ds_sucess['w3'] ))
    
    
    #print("ds_sucess=",ds_sucess)
    
    #print("success=",ds_sucess[(ds_sucess.y==1)])
    
    df11.columns = [c.replace(' ', '_') for c in df11.columns]


    df12=df11[(df11.Publish_Date <= date1 ) & (df11.Publish_Date >= date0 )]

    df12_latest_publish_dt=df12.groupby(['Ticker']).Publish_Date.agg([max])


    #print(df12.columns)

    df12_latest_publish_dt['max_publish_dt']=df12_latest_publish_dt['max']
    df12_tmp = pd.merge(df12, df12_latest_publish_dt, how="inner", on=["Ticker"])
    df13=df12_tmp[df12_tmp.Publish_Date == df12_tmp.max_publish_dt]

    df12_latest_fiscal_yr=df12.groupby(['Ticker']).Fiscal_Year.agg([max])
    df12_latest_fiscal_yr['latest_fiscal_yr']=df12_latest_fiscal_yr['max']
    df13_tmp = pd.merge(df13, df12_latest_fiscal_yr, how="inner", on=["Ticker"])

    df14=df13_tmp[df13_tmp.Fiscal_Year == df13_tmp.latest_fiscal_yr]

    #print(df14.columns)
    df14['tk']=df14.index.get_level_values('Ticker') 

    df15=df14[['tk','Debt_Ratio','Free_Cash_Flow_Per_Share','Sales_Per_Share','Equity_Per_Share']]

    df15['Debt_Per_Share']=df15['Equity_Per_Share']/(1/df15['Debt_Ratio']-1)
    
    #print("df15=", df15)
    
    result1m = pd.merge(ds_sucess, df15, how="inner", on=["tk"])
    
    #result1m = pd.merge(result1l, r1, how="left", on=["tk1"])
    
    
    result1m['x1_p99']=result1m.Free_Cash_Flow_Per_Share.quantile(.99)
    result1m['x2_p99']=result1m.Sales_Per_Share.quantile(.99)
    result1m['x3_p99']=result1m.Equity_Per_Share.quantile(.99)
    result1m['x4_p99']=result1m.Debt_Per_Share.quantile(.99)
    result1m['x5_p99']=result1m.Volume.quantile(.99)
    
    result1m['x1_p1']=result1m.Free_Cash_Flow_Per_Share.quantile(0.01)
    result1m['x2_p1']=result1m.Sales_Per_Share.quantile(0.01)
    result1m['x3_p1']=result1m.Equity_Per_Share.quantile(0.01)
    result1m['x4_p1']=result1m.Debt_Per_Share.quantile(0.01)
    result1m['x5_p1']=result1m.Volume.quantile(.01)
    
    result1n=result1m[(result1m.Volume<=result1m.x5_p99) & (result1m.Free_Cash_Flow_Per_Share>=result1m.x1_p1) & (result1m.Free_Cash_Flow_Per_Share<=result1m.x1_p99) & (result1m.Sales_Per_Share>=result1m.x2_p1) & (result1m.Sales_Per_Share<=result1m.x2_p99) & (result1m.Equity_Per_Share>=result1m.x3_p1) & (result1m.Equity_Per_Share<=result1m.x3_p99) & (result1m.Debt_Per_Share>=result1m.x4_p1) & (result1m.Debt_Per_Share<=result1m.x4_p99)]

    
    result1n['Debt_Per_Share_norm']=result1n['Debt_Per_Share']/result1n['Debt_Per_Share'].mean()
    result1n['Equity_Per_Share_norm']=result1n['Equity_Per_Share']/result1n['Equity_Per_Share'].mean()
    result1n['Sales_Per_Share_norm']=result1n['Sales_Per_Share']/result1n['Sales_Per_Share'].mean()
    result1n['Free_Cash_Flow_Per_Share_norm']=result1n['Free_Cash_Flow_Per_Share']/result1n['Free_Cash_Flow_Per_Share'].mean()
    
    result1n['calc_close_norm']=result1n['calc_close']/result1n['calc_open'].mean()
    
    
#x=traindsoa[['Debt_Per_Share','Equity_Per_Share','Sales_Per_Share','Free_Cash_Flow_Per_Share']]
#odel = ols('calc_close ~ sm.add_constant(x)', data=traindsoa).fit()
    #result1n=result1m
    #have studentized residual based outlier cleaning: jun15 2024
    #print(df15)
    #df13.columns

    
    #result1c['days_to_high']=datetime.datetime.strptime('', "%Y-%m-%d").date()
    #result1d['days_to_high']=result1d['max_dt']-result1d['open_dt']
    #result1d['days_from_high']=result1d['close_dt']-result1d['max_dt']
    
    #result1d['days_to_min']=result1d['min_dt']-result1d['open_dt']
    #result1d['days_from_min']=result1d['close_dt']-result1d['min_dt']
    #result1d['avg_vol']=result1d['tot_vol']/(result1d['close_dt']-result1d['open_dt'])
    
    

    return result1n




    

In [7]:
import sklearn
print(sklearn.__version__)
from sklearn import datasets
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error
from sklearn import neighbors
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from datetime import datetime
import numpy as np

from sklearn.linear_model import LinearRegression

def doe(nn_cutoff, yr1_inp,yr2_inp, p_by_o_high,p_by_o_low,ofile,rego,regp,rego_norm,regp_norm,avg_cpo,avg_x1o,avg_x2o,avg_x3o,avg_x4o, avg_cpn,avg_x1p,avg_x2p,avg_x3p,avg_x4p):
    #ids["p_over_e"]=ids["calc_open"]/ids["Free_Cash_Flow_Per_Share"]
    #ids["profit"]=ids["calc_close"]/ids["calc_open"]-1
    #ids1=ids[ids.p_over_e>0]

    #ids2=ids1.groupby(['dum']).apply(lambda x: x.sort_values(["p_over_e"])).reset_index(drop=True)
    #retds=ids2.groupby('dum').head(15)
    #f1=retds['exp_profit_rego'].mean()

    #print("f1=",f1)
    
    #year1_df1a=pd.read_csv('./finds_07_2021.csv') 
    #year2_df1a=pd.read_csv('./finds_07_2022.csv')
    #year1_df1a=pd.read_csv(yr1_inp) 
    #year2_df1a=pd.read_csv(yr2_inp)
#df_2017=pd.read_csv('../input/jun2222/finds_06_2017_jun22.csv')
#df_2018=pd.read_csv('../input/jun2222/finds_06_2018_jun22.csv')
    #year1_df1a=yr1_inp
    year2_df1a=yr2_inp



    year1_df=yr1_inp

#year2_df1=df_2017.dropna()
    year2_df=year2_df1a[(year2_df1a.calc_open>5) & (year2_df1a.Debt_Per_Share>-999999999999) & (year2_df1a.Debt_Per_Share<999999999999) & (year2_df1a.Equity_Per_Share>-999999999999) & (year2_df1a.Equity_Per_Share<999999999999) & (year2_df1a.Sales_Per_Share>-999999999999) & (year2_df1a.Sales_Per_Share<999999999999) & (year2_df1a.Free_Cash_Flow_Per_Share>-999999999999) & (year2_df1a.Free_Cash_Flow_Per_Share<999999999999)]

    #X_train, X_test1, target, target_test = train_test_split(year1_df[["Debt_Per_Share","Equity_Per_Share", "Sales_Per_Share","Free_Cash_Flow_Per_Share"]], year1_df["calc_close"], random_state=42, test_size=0.01)
    
  
    X_test=year2_df[["Debt_Per_Share","Equity_Per_Share", "Sales_Per_Share","Free_Cash_Flow_Per_Share"]]
    #year2_df["Debt_Per_Share_norm"]=year2_df["Debt_Per_Share"]/avg_x1o
    #year2_df["Equity_Per_Share_norm"]=year2_df["Equity_Per_Share"]/avg_x2o
    #year2_df["Sales_Per_Share_norm"]=year2_df["Sales_Per_Share"]/avg_x3o
    #year2_df["Free_Cash_Flow_Per_Share_norm"]=year2_df["Free_Cash_Flow_Per_Share"]/avg_x4o
    
    X_test_norm=year2_df[["Debt_Per_Share_norm","Equity_Per_Share_norm", "Sales_Per_Share_norm","Free_Cash_Flow_Per_Share_norm"]]
    tkds=year2_df[["tk"]]
        
    p1o=rego.predict(X_test)
    p1o_norm=rego_norm.predict(X_test_norm)
    
    pdfo = pd.DataFrame(p1o, columns = ['reg_closeo'])
    #print("avg_pred=",pdf['reg_close'].mean())
    pdfo['row_num'] = np.arange(len(pdfo))

    pdfo_norm = pd.DataFrame(p1o_norm, columns = ['reg_co1'])
    #print("avg_pred=",pdf['reg_close'].mean())
    pdfo_norm['row_num'] = np.arange(len(pdfo_norm))
    pdfo_norm['reg_closeo1'] =pdfo_norm['reg_co1']*avg_cpo
    
    

    #year2_df["Debt_Per_Share_norm"]=year2_df["Debt_Per_Share"]/avg_x1p
    #year2_df["Equity_Per_Share_norm"]=year2_df["Equity_Per_Share"]/avg_x2p
    #year2_df["Sales_Per_Share_norm"]=year2_df["Sales_Per_Share"]/avg_x3p
    #year2_df["Free_Cash_Flow_Per_Share_norm"]=year2_df["Free_Cash_Flow_Per_Share"]/avg_x4p
    #X_test_norm=year2_df[["Debt_Per_Share_norm","Equity_Per_Share_norm", "Sales_Per_Share_norm","Free_Cash_Flow_Per_Share_norm"]]

    
    p1p=regp.predict(X_test)
    p1p_norm=regp_norm.predict(X_test_norm)
    
    pdfp = pd.DataFrame(p1p, columns = ['reg_closep'])
    #print("avg_pred=",pdf['reg_close'].mean())
    pdfp['row_num'] = np.arange(len(pdfp))

    pdfp_norm = pd.DataFrame(p1p_norm, columns = ['reg_cp1'])
    #print("avg_pred=",pdf['reg_close'].mean())
    pdfp_norm['row_num'] = np.arange(len(pdfp_norm))
    pdfp_norm['reg_closep1'] =pdfp_norm['reg_cp1']*avg_cpp
    

    X_test['row_num'] = np.arange(len(X_test))
    regdsto1 = pd.merge(X_test, pdfo, on=["row_num"])
    regdstp1 = pd.merge(regdsto1, pdfp, on=["row_num"])
    regdsto = pd.merge(regdstp1, pdfo_norm, on=["row_num"])
    regdstp = pd.merge(regdsto, pdfp_norm, on=["row_num"])
    tkds['row_num'] = np.arange(len(tkds))
    regds = pd.merge(regdstp, tkds, on=["row_num"])
    #print("regds=",regds)

    #regds.to_csv('regds.csv')
    
    #pdf1 = pd.DataFrame(p1, columns = ['pred_mthsales'])
            

#year1_df['std_open']=year1_df['Open'].std()
    year1_df['dol_vol_yr1']=year1_df['Volume']*year1_df["calc_open"]
    
    year1_df['std_Free_Cash_Flow']=year1_df['Free_Cash_Flow_Per_Share'].std()
    year1_df['std_Sales_Per_Share']=year1_df['Sales_Per_Share'].std()
    year1_df['std_Equity_Per_Share']=year1_df['Equity_Per_Share'].std()
    year1_df['std_Debt_Per_Share']=year1_df['Debt_Per_Share'].std()
    year1_df['std_vol']=year1_df['Volume'].std()
    year1_df['std_dol_vol']=year1_df['dol_vol_yr1'].std()

    year1_df["Free_Cash_Flow_Per_Share_yr1"]=year1_df["Free_Cash_Flow_Per_Share"]
    year1_df[ "Sales_Per_Share_yr1"]=year1_df[ "Sales_Per_Share"]
    year1_df["Equity_Per_Share_yr1"]=year1_df[ "Equity_Per_Share"]
    year1_df[ "Debt_Per_Share_yr1"]=year1_df["Debt_Per_Share"]
    year1_df[ "Volume_yr1"]=year1_df["Volume"]
    year1_df[ "open_yr1"]=year1_df["calc_open"]
    year1_df[ "close_yr1"]=year1_df["calc_close"]
    year1_df[ "dummy"]='D'
    year1_df[ "y1"]=year1_df["y"]

    year2_df['dol_vol_yr2']=year2_df['Volume']*year2_df["calc_open"]
    year2_df["Free_Cash_Flow_Per_Share_yr2"]=year2_df["Free_Cash_Flow_Per_Share"]
    year2_df[ "Sales_Per_Share_yr2"]=year2_df[ "Sales_Per_Share"]
    year2_df["Equity_Per_Share_yr2"]=year2_df[ "Equity_Per_Share"]
    year2_df[ "Debt_Per_Share_yr2"]=year2_df["Debt_Per_Share"]
    year2_df[ "Volume_yr2"]=year2_df["Volume"]
    year2_df[ "dummy"]='D'
    year2_df[ "open_yr2"]=year2_df["calc_open"]
    year2_df[ "close_yr2"]=year2_df["calc_close"]

    year2_df[ "y2"]=year2_df["y"]

   


    magnitude_trainds=year1_df[[ "Sector1", "y1","dol_vol_yr1", "Volume_yr1","open_yr1","close_yr1","std_dol_vol" ,"std_vol", "std_Debt_Per_Share","std_Free_Cash_Flow","std_Sales_Per_Share","std_Equity_Per_Share","Free_Cash_Flow_Per_Share_yr1", "Sales_Per_Share_yr1", "Equity_Per_Share_yr1", "Debt_Per_Share_yr1"]]
    #print("magnitude_trainds",magnitude_trainds)

    magnitude_testds=year2_df[["tk", "Sector1","y2","dol_vol_yr2", "Volume_yr2","open_yr2","close_yr2","Free_Cash_Flow_Per_Share_yr2", "Sales_Per_Share_yr2", "Equity_Per_Share_yr2", "Debt_Per_Share_yr2"]]
    #print("magnitude_test",magnitude_testds)


#magnitude_testds1=magnitude_testds[(magnitude_testds.tk1=='GOOG') | (magnitude_testds.tk1=='AAL') | (magnitude_testds.tk1=='AMZN')]
#magnitude_testds1=magnitude_testds
#.33*2*abs(magnitude_cartprod['Debt_Per_Share_yr2']-magnitude_cartprod['Debt_Per_Share_yr1'])/magnitude_cartprod['std_Debt_Per_Share']+
    magnitude_cartprod=pd.merge(magnitude_testds, magnitude_trainds, how="inner", on=["Sector1"])
    magnitude_cartprod['magnitude_dist']=.33*2*0*abs(magnitude_cartprod['dol_vol_yr2']-magnitude_cartprod['dol_vol_yr1'])/magnitude_cartprod['std_dol_vol']+.33*2*abs(magnitude_cartprod['Debt_Per_Share_yr2']-magnitude_cartprod['Debt_Per_Share_yr1'])/magnitude_cartprod['std_Debt_Per_Share']+.33*2*abs(magnitude_cartprod['Equity_Per_Share_yr2']-magnitude_cartprod['Equity_Per_Share_yr1'])/magnitude_cartprod['std_Equity_Per_Share']+.33*2*abs(magnitude_cartprod['Sales_Per_Share_yr2']-magnitude_cartprod['Sales_Per_Share_yr1'])/magnitude_cartprod['std_Sales_Per_Share']+.33*2*abs(magnitude_cartprod['Free_Cash_Flow_Per_Share_yr2']-magnitude_cartprod['Free_Cash_Flow_Per_Share_yr1'])/magnitude_cartprod['std_Free_Cash_Flow']
#magnitude_cartprod['magnitude_dist']=9999 if isnull(magnitude_cartprod.magnitude_dist) else magnitude_cartprod['magnitude_dist'];
    magnitude_cartprod['null_mag_dist']=magnitude_cartprod['magnitude_dist'].isnull()
    magnitude_cartprod_a=magnitude_cartprod[magnitude_cartprod.null_mag_dist==True]
    magnitude_cartprod_a['magnitude_dist']=99999

    magnitude_cartprod_b=magnitude_cartprod[magnitude_cartprod.null_mag_dist==False]

    #print("maga=",magnitude_cartprod_a)
    #print("magb=",magnitude_cartprod_b)
    magnitude_cartprod=magnitude_cartprod_a.append(magnitude_cartprod_b)

    magnitude_cartprod=magnitude_cartprod[magnitude_cartprod.magnitude_dist<=5000.3]
    #print("magd=",magnitude_cartprod)
    
    

 
#"year_high_yr2","year_min_yr2","open_yr2","close_yr2"

    df13=magnitude_cartprod.groupby(['tk']).apply(lambda x: x.sort_values(["magnitude_dist"])).reset_index(drop=True)
    df14=df13.groupby('tk').head(nn_cutoff)
#df14['adj_open_yr1']=df14['open_yr1']*df14['Sector_avg_yr2']/df14['Sector_avg_yr1']
    #print(df14)

    df17b=df14.groupby(['tk']).y1.agg([sum])
    df17b['avg_year_y1']=df17b['sum']/nn_cutoff

    df17c=df14.groupby(['tk']).close_yr1.agg([sum])
    df17c['close_yr1_nn']=df17c['sum']/nn_cutoff
    
    df17d=df14.groupby(['tk']).open_yr1.agg([sum])
    df17d['open_yr1_nn']=df17d['sum']/nn_cutoff



    df40a=pd.merge(year2_df, df17b, how="inner", on=["tk"])
    df40b=pd.merge(df40a, df17c, how="inner", on=["tk"])
    
    df40c=pd.merge(df40b, regds, how="inner", on=["tk"])
    
    df40=pd.merge(df40c, df17d, how="inner", on=["tk"])
    
    df40['exp_profit_nn']=df40['close_yr1_nn']/df40['open_yr2']-1
    df40['Nrn']=df40['exp_profit_nn']*df40['Volume']
    
    df40['exp_profit_rego']=df40['reg_closeo']/df40['open_yr2']-1
    df40['Nro']=df40['exp_profit_rego']*df40['Volume']

    df40['exp_profit_rego1']=df40['reg_closeo1']/df40['open_yr2']-1
    df40['Nro1']=df40['exp_profit_rego1']*df40['Volume']

    
    df40['exp_profit_regp']=df40['reg_closep']/df40['open_yr2']-1
    df40['Nrp']=df40['exp_profit_regp']*df40['Volume']

    df40['exp_profit_regp1']=df40['reg_closep1']/df40['open_yr2']-1
    df40['Nrp1']=df40['exp_profit_regp1']*df40['Volume']

    
    df40['exp_profit_ens']=0.33333*(df40['reg_closeo']+df40['reg_closep']+df40['close_yr1_nn'])/df40['open_yr2']-1
    df40['Nre']=df40['exp_profit_ens']*df40['Volume']
    
    df40['buy_open_gt1_nn']=df40['open_yr1_nn']/df40['open_yr2']-1


#p_by_o_high=1.4
#p_by_o_low=0.8

#df40=df40.groupby(['dummy']).apply(lambda x: x.sort_values(["avg_year_y1"])).reset_index(drop=True)

    df40.loc[df40['y2'] ==1, 'sp'] = df40['open_yr2']*p_by_o_high
    df40.loc[df40['y2'] ==0, 'sp'] = df40['close_yr2']

    df40['act_profit_b']=df40['sp']/df40['open_yr2']-1
    df40['act_profit_r']=df40['close_yr2']/df40['open_yr2']-1
    
    df40['Nr2']=df40['act_profit_b']*df40['Volume']
    df40['Nr3']=df40['act_profit_r']*df40['Volume']
    
    df40['high_cut']=(p_by_o_high-1)*100
    df40['low_cut']=(1-p_by_o_low)*100
    df40['nn_ct']=nn_cutoff
    df40.to_csv('./df40_oct5.csv')
    #p_by_o_high=1.4
#p_by_o_low=0.8
    #print("df40=",df40)
   # df40.to_csv(ofile)
    df40a=df40[(df40.avg_year_y1>=.6)]
    df40a['dum']='D'
    df40a['exp_profit_ens1']=-df40a['exp_profit_ens']
    df41=df40a.groupby(['dum']).apply(lambda x: x.sort_values(["exp_profit_ens1"])).reset_index(drop=True)
    df42=df41.groupby('dum').head(15)
    
    #print("mean exp reg ens wt_profit=",df40['exp_profit_reg'].mean(), df40['exp_profit_ens'].mean(), df40['Nr'].sum()/df40['Volume'].sum())
    #print("mean act full=",df40['act_profit_b'].mean(), df40['act_profit_r'].mean())
    #print("mean act cut urs=",df42['act_profit_b'].mean(), df42['act_profit_r'].mean())
    
    f1=df40['exp_profit_rego'].mean()
    f2=df40['Nro'].sum()/df40['Volume'].sum()

    f1a=df40['exp_profit_rego1'].mean()
    f2a=df40['Nro1'].sum()/df40['Volume'].sum()
    
    f3=df40['exp_profit_regp'].mean()
    f4=df40['Nrp'].sum()/df40['Volume'].sum()

    f3a=df40['exp_profit_regp1'].mean()
    f4a=df40['Nrp1'].sum()/df40['Volume'].sum()
    
    
    f5=df40['exp_profit_nn'].mean()
    f6=df40['Nrn'].sum()/df40['Volume'].sum()
    
    f7=df40['act_profit_b'].mean()
    f8=df40['act_profit_r'].mean()
    f9=df42['act_profit_b'].mean()
    f10=df42['act_profit_r'].mean()
    f11=df40['Nr2'].sum()/df40['Volume'].sum()
    f12=df40['Nr2'].sum()/df40['Volume'].sum()
    
    f13=df40['exp_profit_ens'].mean()
    f14=df40['Nre'].sum()/df40['Volume'].sum()
    
    data = {'exp_profit_rego': [f1],
            'wt_profit_rego': [f2],

            'exp_profit_rego1': [f1a],
            'wt_profit_rego1': [f2a],
            
            'exp_profit_regp': [f3],
            'wt_profit_regp': [f4],

            'exp_profit_regp1': [f3a],
            'wt_profit_regp1': [f4a],
            
            'exp_profit_nn': [f5],
            'wt_profit_nn': [f6],
            
            'exp_profit_ens': [f13],
            'wt_profit_ens': [f14],
            
            'clipped_act_full': [f7],
            'unclipped_act_full': [f8],
            'clipped_act_15': [f9],
            'unclipped_act_15': [f10],
            'clipped_wt_actfull': [f11],
            'unclipped_wt_actfull': [f12]
           }
    
    #data = {f1, f2, f3,f4, f5, f6, f7}
 
# Create the pandas DataFrame
    #retdf = pd.DataFrame(data, columns=['exp_profit_reg', 'exp_profit_ens', 'wt_profit', 'clipped_act_full', 'unclipped_act_full', 'clipped_act_15', 'unclipped_act_15'])
    retdf = pd.DataFrame(data)

    
    
    
    
    return retdf



/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


1.2.2


In [8]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

#data for estimating relationship between company fundamentals and closing price in 1 year-optimistic 1; remove outliers

from statsmodels.stats.outliers_influence import outlier_test

dt1o='2009-03-02' 
dt2o='2010-03-02'
dt0o='2008-03-01'

traindso1=get_current_state(dt1o, dt2o,dt0o,1.4,0.7)

#results=sm.OLS(y, sm.add_constant(x)).fit()

#traindsoa=traindso1[['Volume','calc_close','Debt_Per_Share','Equity_Per_Share','Sales_Per_Share','Free_Cash_Flow_Per_Share']]
traindsoa=traindso1
x=traindsoa[['Debt_Per_Share','Equity_Per_Share','Sales_Per_Share','Free_Cash_Flow_Per_Share']]
model = ols('calc_close ~ sm.add_constant(x)', data=traindsoa).fit()

print("traindsoa=",traindsoa)
stud_res = model.outlier_test()

stud_res['row_num'] = np.arange(len(stud_res))

traindsoa['row_num'] = np.arange(len(traindsoa['calc_close']))
traindsob = pd.merge(traindsoa, stud_res, on=["row_num"])

traindso=traindsob[(traindsob.student_resid>-3) & (traindsob.student_resid<3)]

print("traindso=",traindso)




/tmp/ipykernel_19/186761998.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_yr1['calc_price']=0.5*(df_yr1['Low']+df_yr1['High'])
/tmp/ipykernel_19/186761998.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_yr1_open1['calc_open']=df_yr1_open1['calc_price']
/tmp/ipykernel_19/186761998.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pyda

actual date1= 2009-03-02
df_yr1_open=                      tk  calc_open    Volume
Ticker Date                                 
A      2009-03-02     A     13.155   4288200
AACG   2009-03-02  AACG      5.230      9900
AAIC   2009-03-02  AAIC      2.600     40900
AAL    2009-03-02   AAL      2.640  17207507
AAME   2009-03-02  AAME      0.640      1500
...                 ...        ...       ...
ZEUS   2009-03-02  ZEUS     11.535    267100
ZION   2009-03-02  ZION      9.140   5502400
ZIXI   2009-03-02  ZIXI      1.005    240400
ZNOG   2009-03-02  ZNOG     13.420    107100
ZUMZ   2009-03-02  ZUMZ      8.020    317800

[2677 rows x 3 columns]
actual date2= 2010-03-02
df_yr1_close=                      tk         dt  calc_close
Ticker Date                                   
A      2010-03-02     A 2010-03-02      32.195
AACG   2010-03-02  AACG 2010-03-02       3.810
AAIC   2010-03-02  AAIC 2010-03-02      18.290
AAL    2010-03-02   AAL 2010-03-02       7.660
AAME   2010-03-02  AAME 2010-03

/tmp/ipykernel_19/186761998.py:150: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df14['tk']=df14.index.get_level_values('Ticker')
/tmp/ipykernel_19/186761998.py:154: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df15['Debt_Per_Share']=df15['Equity_Per_Share']/(1/df15['Debt_Ratio']-1)
/tmp/ipykernel_19/186761998.py:178: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: htt

traindsoa=                 tk  calc_open    Volume  max_x  max_w1  max_y  max_w2  \
1             ABAX     14.950    416800    1.0     1.0    0.0     0.0   
3             ABMD      6.345    468100    1.0     1.0    0.0     0.0   
4              ABT     46.395  33721100    0.0     0.0    0.0     0.0   
6    ACET_delisted      5.835    179200    0.0     0.0    0.0     0.0   
8             ACIW      5.940   2285400    0.0     0.0    0.0     0.0   
..             ...        ...       ...    ...     ...    ...     ...   
707            XOM     66.135  54827200    0.0     0.0    0.0     0.0   
709            XPO      1.155     23131    1.0     1.0    0.0     0.0   
710              Y    244.640     37461    0.0     0.0    0.0     0.0   
713            ZBH     33.095   4307048    1.0     1.0    0.0     0.0   
714           ZIXI      1.005    240400    1.0     1.0    0.0     0.0   

         min_x  min_gp_dt min_y  ...      x1_p1     x2_p1     x3_p1  \
1   2009-07-30 2009-07-30   NaT  ... -28.

In [9]:

#regression for estimating relationship between company fundamentals and closing price in 1 year-optimistic 1

#dt1o='2009-03-02' 
#dt2o='2012-03-02'
#dt0o='2008-03-01'



#traindso1=get_current_state(dt1o, dt2o,dt0o,1.4,0.7)
#traindso1['clpe']=traindso1['calc_close']/traindso1['Free_Cash_Flow_Per_Share']
#traindso=traindso1[(traindso1.calc_open>5) & (traindso1.clpe>-800) & (traindso1.clpe<800) & (traindso1.Debt_Per_Share>-9999999999) & (traindso1.Debt_Per_Share<9999999999) & (traindso1.Equity_Per_Share>-9999999999) & (traindso1.Equity_Per_Share<9999999999) & (traindso1.Sales_Per_Share>-9999999999) & (traindso1.Sales_Per_Share<9999999999) & (traindso1.Free_Cash_Flow_Per_Share>-9999999999) & (traindso1.Free_Cash_Flow_Per_Share<9999999999)]
#print("year1_df=",year1_df.head())


#traindso=traindso1
traindso['Sector1']='dum'



print("traindso=",traindso)

X_traino=traindso[["Debt_Per_Share","Equity_Per_Share", "Sales_Per_Share","Free_Cash_Flow_Per_Share"]]
#X_traino=traindso[["Equity_Per_Share", "Free_Cash_Flow_Per_Share"]]    
target=traindso["calc_close"]
reg_o  = sklearn.linear_model.LinearRegression()



   
reg_o.fit(X_traino, target)

regdfo=pd.DataFrame(zip(X_traino.columns, reg_o.coef_))
print("regdfo=",regdfo)

print("o intercept=",reg_o.intercept_)
    
traindso1.to_csv('./traindso1.csv')
avg_cpo=traindso["calc_close"].mean()
avg_x1o=traindso["Debt_Per_Share"].mean()
avg_x2o=traindso["Equity_Per_Share"].mean()
avg_x3o=traindso["Sales_Per_Share"].mean()
avg_x4o=traindso["Free_Cash_Flow_Per_Share"].mean()


traindso=                 tk  calc_open    Volume  max_x  max_w1  max_y  max_w2  \
0             ABAX     14.950    416800    1.0     1.0    0.0     0.0   
1             ABMD      6.345    468100    1.0     1.0    0.0     0.0   
2              ABT     46.395  33721100    0.0     0.0    0.0     0.0   
3    ACET_delisted      5.835    179200    0.0     0.0    0.0     0.0   
4             ACIW      5.940   2285400    0.0     0.0    0.0     0.0   
..             ...        ...       ...    ...     ...    ...     ...   
593            XEL     17.415   3919300    0.0     0.0    0.0     0.0   
594            XOM     66.135  54827200    0.0     0.0    0.0     0.0   
595            XPO      1.155     23131    1.0     1.0    0.0     0.0   
597            ZBH     33.095   4307048    1.0     1.0    0.0     0.0   
598           ZIXI      1.005    240400    1.0     1.0    0.0     0.0   

         min_x  min_gp_dt min_y  ... Debt_Per_Share_norm  \
0   2009-07-30 2009-07-30   NaT  ...            0.000

/tmp/ipykernel_19/1396489655.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  traindso['Sector1']='dum'


In [10]:
#data for estimating relationship between company fundamentals and closing price in 1 year-optimistic version 2; remove outliers
#import statsmodels.api as sm
#from statsmodels.formula.api import ols

#from statsmodels.stats.outliers_influence import outlier_test

#dt1o='2009-03-02' 
#dt2o='2010-03-02'
#dt0o='2008-03-01'

#traindso1=get_current_state(dt1o, dt2o,dt0o,1.4,0.7)

#results=sm.OLS(y, sm.add_constant(x)).fit()

#traindsoa=traindso1[['Volume','calc_close','Debt_Per_Share','Equity_Per_Share','Sales_Per_Share','Free_Cash_Flow_Per_Share']]
#traindsoa=traindso
traindsoan=traindso1
xn=traindsoan[['Debt_Per_Share_norm','Equity_Per_Share_norm','Sales_Per_Share_norm','Free_Cash_Flow_Per_Share_norm']]
modeln = ols('calc_close_norm ~ sm.add_constant(xn)', data=traindsoan).fit()

print("traindsoan=",traindsoan)
stud_resn = modeln.outlier_test()

stud_resn['row_num'] = np.arange(len(stud_resn))

traindsoan['row_num'] = np.arange(len(traindsoan['calc_close_norm']))
traindsobn = pd.merge(traindsoan, stud_resn, on=["row_num"])


traindson=traindsobn[(traindsobn.student_resid>-3) & (traindsobn.student_resid<3)]

print("traindson=",traindson)


traindsoan=                 tk  calc_open    Volume  max_x  max_w1  max_y  max_w2  \
1             ABAX     14.950    416800    1.0     1.0    0.0     0.0   
3             ABMD      6.345    468100    1.0     1.0    0.0     0.0   
4              ABT     46.395  33721100    0.0     0.0    0.0     0.0   
6    ACET_delisted      5.835    179200    0.0     0.0    0.0     0.0   
8             ACIW      5.940   2285400    0.0     0.0    0.0     0.0   
..             ...        ...       ...    ...     ...    ...     ...   
707            XOM     66.135  54827200    0.0     0.0    0.0     0.0   
709            XPO      1.155     23131    1.0     1.0    0.0     0.0   
710              Y    244.640     37461    0.0     0.0    0.0     0.0   
713            ZBH     33.095   4307048    1.0     1.0    0.0     0.0   
714           ZIXI      1.005    240400    1.0     1.0    0.0     0.0   

         min_x  min_gp_dt min_y  ...     x2_p1     x3_p1      x4_p1  x5_p1  \
1   2009-07-30 2009-07-30   NaT  

In [11]:
#regression for estimating relationship between company fundamentals and closing price in 1 year-optimistic version 2; 

#traindso1=get_current_state(dt1o, dt2o,dt0o,1.4,0.7)
#traindso1['clpe']=traindso1['calc_close']/traindso1['Free_Cash_Flow_Per_Share']
#traindso=traindso1[(traindso1.calc_open>5) & (traindso1.clpe>-800) & (traindso1.clpe<800) & (traindso1.Debt_Per_Share>-9999999999) & (traindso1.Debt_Per_Share<9999999999) & (traindso1.Equity_Per_Share>-9999999999) & (traindso1.Equity_Per_Share<9999999999) & (traindso1.Sales_Per_Share>-9999999999) & (traindso1.Sales_Per_Share<9999999999) & (traindso1.Free_Cash_Flow_Per_Share>-9999999999) & (traindso1.Free_Cash_Flow_Per_Share<9999999999)]
#print("year1_df=",year1_df.head())


#traindso=traindso1
traindson['Sector1']='dum'



print("traindson=",traindson)

X_trainon=traindson[["Debt_Per_Share_norm","Equity_Per_Share_norm", "Sales_Per_Share_norm","Free_Cash_Flow_Per_Share_norm"]]
#X_traino=traindso[["Equity_Per_Share", "Free_Cash_Flow_Per_Share"]]    
targetn=traindson["calc_close_norm"]
reg_on  = sklearn.linear_model.LinearRegression()



   
reg_on.fit(X_trainon, targetn)

regdfon=pd.DataFrame(zip(X_trainon.columns, reg_on.coef_))
print("regdfon=",regdfon)

print("o intercept=",reg_on.intercept_)
    
#traindso1n.to_csv('./traindso1n.csv')

traindson=                 tk  calc_open    Volume  max_x  max_w1  max_y  max_w2  \
0             ABAX     14.950    416800    1.0     1.0    0.0     0.0   
1             ABMD      6.345    468100    1.0     1.0    0.0     0.0   
2              ABT     46.395  33721100    0.0     0.0    0.0     0.0   
3    ACET_delisted      5.835    179200    0.0     0.0    0.0     0.0   
4             ACIW      5.940   2285400    0.0     0.0    0.0     0.0   
..             ...        ...       ...    ...     ...    ...     ...   
593            XEL     17.415   3919300    0.0     0.0    0.0     0.0   
594            XOM     66.135  54827200    0.0     0.0    0.0     0.0   
595            XPO      1.155     23131    1.0     1.0    0.0     0.0   
597            ZBH     33.095   4307048    1.0     1.0    0.0     0.0   
598           ZIXI      1.005    240400    1.0     1.0    0.0     0.0   

         min_x  min_gp_dt min_y  ... Debt_Per_Share_norm  \
0   2009-07-30 2009-07-30   NaT  ...            0.00

/tmp/ipykernel_19/2854002092.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  traindson['Sector1']='dum'


In [12]:
#data for estimating relationship between company fundamentals and closing price in 1 year-pessimistic version 1; remove outliers

import statsmodels.api as sm
from statsmodels.formula.api import ols

from statsmodels.stats.outliers_influence import outlier_test


dt0p='2006-10-01' 
dt1p='2007-10-08'
dt2p='2008-10-07'

traindsp1=get_current_state(dt1p, dt2p,dt0p,1.4,0.7)

#results=sm.OLS(y, sm.add_constant(x)).fit()

#traindspa=traindsp1[['Volume','calc_close','Debt_Per_Share','Equity_Per_Share','Sales_Per_Share','Free_Cash_Flow_Per_Share']]
traindspa=traindsp1
x=traindspa[['Debt_Per_Share','Equity_Per_Share','Sales_Per_Share','Free_Cash_Flow_Per_Share']]
model = ols('calc_close ~ sm.add_constant(x)', data=traindspa).fit()

print("traindspa=",traindspa)
stud_resp = model.outlier_test()

stud_resp['row_num'] = np.arange(len(stud_resp))

traindspa['row_num'] = np.arange(len(traindspa['calc_close']))
traindspb = pd.merge(traindspa, stud_resp, on=["row_num"])

traindsp=traindspb[(traindspb.student_resid>-3) & (traindspb.student_resid<3)]

print("traindsp=",traindsp)


avg_cpp=traindsp["calc_close"].mean()

avg_x1p=traindsp["Debt_Per_Share"].mean()
avg_x2p=traindsp["Equity_Per_Share"].mean()
avg_x3p=traindsp["Sales_Per_Share"].mean()
avg_x4p=traindsp["Free_Cash_Flow_Per_Share"].mean()

traindsp.to_csv('./traindsp.csv')

/tmp/ipykernel_19/186761998.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_yr1['calc_price']=0.5*(df_yr1['Low']+df_yr1['High'])
/tmp/ipykernel_19/186761998.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_yr1_open1['calc_open']=df_yr1_open1['calc_price']
/tmp/ipykernel_19/186761998.py:61: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pyda

actual date1= 2007-10-08
df_yr1_open=                      tk  calc_open   Volume
Ticker Date                                
A      2007-10-08     A     37.810   999000
AAIC   2007-10-08  AAIC     95.400    48600
AAL    2007-10-08   AAL     30.345  4918231
AAME   2007-10-08  AAME      2.675     3300
AAON   2007-10-08  AAON      4.065   172201
...                 ...        ...      ...
ZEUS   2007-10-08  ZEUS     28.200   236300
ZION   2007-10-08  ZION     71.865   524000
ZIXI   2007-10-08  ZIXI      2.445  1263300
ZNOG   2007-10-08  ZNOG      6.685    40700
ZUMZ   2007-10-08  ZUMZ     47.310   316000

[2561 rows x 3 columns]
actual date2= 2008-10-07
df_yr1_close=                      tk         dt  calc_close
Ticker Date                                   
A      2008-10-07     A 2008-10-07      26.280
AACG   2008-10-07  AACG 2008-10-07       7.430
AAIC   2008-10-07  AAIC 2008-10-07      21.300
AAL    2008-10-07   AAL 2008-10-07       5.625
AAME   2008-10-07  AAME 2008-10-07       0.7

/tmp/ipykernel_19/186761998.py:150: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df14['tk']=df14.index.get_level_values('Ticker')
/tmp/ipykernel_19/186761998.py:154: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df15['Debt_Per_Share']=df15['Equity_Per_Share']/(1/df15['Debt_Ratio']-1)
/tmp/ipykernel_19/186761998.py:178: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: htt

traindspa=                 tk  calc_open    Volume  max_x  max_w1  max_y  max_w2  \
1             ABAX     25.235    376300    1.0     1.0    1.0     1.0   
2             ABMD     13.130    173600    1.0     1.0    0.0     0.0   
3              ABT     55.035   5505200    0.0     0.0    0.0     0.0   
5    ACET_delisted      9.290     34700    0.0     0.0    1.0     1.0   
8             ACTG     15.890     70500    0.0     0.0    1.0     1.0   
..             ...        ...       ...    ...     ...    ...     ...   
626            XEL     22.030   1139800    0.0     0.0    0.0     0.0   
627            XOM     90.490  12833200    0.0     0.0    0.0     0.0   
629            XPO      1.710     39901    0.0     0.0    0.0     0.0   
633            ZBH     80.320    715129    0.0     0.0    1.0     1.0   
634           ZIXI      2.445   1263300    1.0     1.0    0.0     0.0   

         min_x  min_gp_dt      min_y  ...      x1_p1     x2_p1     x3_p1  \
1   2007-12-10 2007-12-10 2008-10-06

In [13]:
#regression for estimating relationship between company fundamentals and closing price in 1 year-pessimistic version 1; 


#traindsp1=get_current_state(dt1p, dt2p,dt0p,1.4,0.7)
#traindsp1['clpe']=traindsp1['calc_close']/traindsp1['Free_Cash_Flow_Per_Share']
#traindsp=traindsp1[(traindsp1.calc_open>5) & (traindsp1.clpe>-800) & (traindsp1.clpe<800) & (traindsp1.Debt_Per_Share>-9999999999) & (traindsp1.Debt_Per_Share<9999999999) & (traindsp1.Equity_Per_Share>-9999999999) & (traindsp1.Equity_Per_Share<9999999999) & (traindsp1.Sales_Per_Share>-9999999999) & (traindsp1.Sales_Per_Share<9999999999) & (traindsp1.Free_Cash_Flow_Per_Share>-9999999999) & (traindsp1.Free_Cash_Flow_Per_Share<9999999999)]
#print("year1_df=",year1_df.head())


traindsp['Sector1']='dum'

print("traindsp=",traindsp)

X_trainp=traindsp[["Debt_Per_Share","Equity_Per_Share", "Sales_Per_Share","Free_Cash_Flow_Per_Share"]]
    
target=traindsp["calc_close"]
reg_p  = sklearn.linear_model.LinearRegression()

   
reg_p.fit(X_trainp, target)

regdfp=pd.DataFrame(zip(X_trainp.columns, reg_p.coef_))
print("regdfp=",regdfp)

print("p intercept=",reg_p.intercept_)

trainds=traindso.append(traindsp)


traindsp=                 tk  calc_open    Volume  max_x  max_w1  max_y  max_w2  \
0             ABAX     25.235    376300    1.0     1.0    1.0     1.0   
1             ABMD     13.130    173600    1.0     1.0    0.0     0.0   
2              ABT     55.035   5505200    0.0     0.0    0.0     0.0   
3    ACET_delisted      9.290     34700    0.0     0.0    1.0     1.0   
4             ACTG     15.890     70500    0.0     0.0    1.0     1.0   
..             ...        ...       ...    ...     ...    ...     ...   
512            XEL     22.030   1139800    0.0     0.0    0.0     0.0   
513            XOM     90.490  12833200    0.0     0.0    0.0     0.0   
514            XPO      1.710     39901    0.0     0.0    0.0     0.0   
515            ZBH     80.320    715129    0.0     0.0    1.0     1.0   
516           ZIXI      2.445   1263300    1.0     1.0    0.0     0.0   

         min_x  min_gp_dt      min_y  ... Debt_Per_Share_norm  \
0   2007-12-10 2007-12-10 2008-10-06  ...       

/tmp/ipykernel_19/3875444703.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  traindsp['Sector1']='dum'
/tmp/ipykernel_19/3875444703.py:27: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  trainds=traindso.append(traindsp)


In [14]:
#data for estimating relationship between company fundamentals and closing price in 1 year-pessimistic version 2; remove outliers

#import statsmodels.api as sm
#from statsmodels.formula.api import ols

#from statsmodels.stats.outliers_influence import outlier_test


#dt0p='2006-10-01' 
#dt1p='2007-10-08'
#dt2p='2008-10-07'

#traindsp1=get_current_state(dt1p, dt2p,dt0p,1.4,0.7)

#results=sm.OLS(y, sm.add_constant(x)).fit()

#traindspa=traindsp1[['Volume','calc_close','Debt_Per_Share','Equity_Per_Share','Sales_Per_Share','Free_Cash_Flow_Per_Share']]
traindspan=traindsp1
xn=traindspan[['Debt_Per_Share_norm','Equity_Per_Share_norm','Sales_Per_Share_norm','Free_Cash_Flow_Per_Share_norm']]
modeln = ols('calc_close_norm ~ sm.add_constant(xn)', data=traindspan).fit()

print("traindspan=",traindspan)
stud_respn = modeln.outlier_test()

stud_respn['row_num'] = np.arange(len(stud_respn))

traindspan['row_num'] = np.arange(len(traindspan['calc_close_norm']))
traindspbn = pd.merge(traindspan, stud_respn, on=["row_num"])

traindspn=traindspbn[(traindspbn.student_resid>-3) & (traindspbn.student_resid<3)]


print("traindspn=",traindspn)

#traindsp.to_csv('./traindsp.csv')


traindspan=                 tk  calc_open    Volume  max_x  max_w1  max_y  max_w2  \
1             ABAX     25.235    376300    1.0     1.0    1.0     1.0   
2             ABMD     13.130    173600    1.0     1.0    0.0     0.0   
3              ABT     55.035   5505200    0.0     0.0    0.0     0.0   
5    ACET_delisted      9.290     34700    0.0     0.0    1.0     1.0   
8             ACTG     15.890     70500    0.0     0.0    1.0     1.0   
..             ...        ...       ...    ...     ...    ...     ...   
626            XEL     22.030   1139800    0.0     0.0    0.0     0.0   
627            XOM     90.490  12833200    0.0     0.0    0.0     0.0   
629            XPO      1.710     39901    0.0     0.0    0.0     0.0   
633            ZBH     80.320    715129    0.0     0.0    1.0     1.0   
634           ZIXI      2.445   1263300    1.0     1.0    0.0     0.0   

         min_x  min_gp_dt      min_y  ...     x2_p1     x3_p1    x4_p1  x5_p1  \
1   2007-12-10 2007-12-10 2008

In [15]:

#regression for estimating relationship between company fundamentals and closing price in 1 year-pessimistic version 2; 

#traindsp1=get_current_state(dt1p, dt2p,dt0p,1.4,0.7)
#traindsp1['clpe']=traindsp1['calc_close']/traindsp1['Free_Cash_Flow_Per_Share']
#traindsp=traindsp1[(traindsp1.calc_open>5) & (traindsp1.clpe>-800) & (traindsp1.clpe<800) & (traindsp1.Debt_Per_Share>-9999999999) & (traindsp1.Debt_Per_Share<9999999999) & (traindsp1.Equity_Per_Share>-9999999999) & (traindsp1.Equity_Per_Share<9999999999) & (traindsp1.Sales_Per_Share>-9999999999) & (traindsp1.Sales_Per_Share<9999999999) & (traindsp1.Free_Cash_Flow_Per_Share>-9999999999) & (traindsp1.Free_Cash_Flow_Per_Share<9999999999)]
#print("year1_df=",year1_df.head())


traindspn['Sector1']='dum'

print("traindspn=",traindspn)

X_trainpn=traindspn[["Debt_Per_Share_norm","Equity_Per_Share_norm", "Sales_Per_Share_norm","Free_Cash_Flow_Per_Share_norm"]]
    
targetn=traindspn["calc_close_norm"]
reg_pn  = sklearn.linear_model.LinearRegression()

   
reg_pn.fit(X_trainpn, targetn)

regdfpn=pd.DataFrame(zip(X_trainpn.columns, reg_pn.coef_))
print("regdfpn=",regdfpn)

print("p intercept n=",reg_pn.intercept_)

#trainds=traindso.append(traindsp)

traindspn=                 tk  calc_open    Volume  max_x  max_w1  max_y  max_w2  \
0             ABAX     25.235    376300    1.0     1.0    1.0     1.0   
1             ABMD     13.130    173600    1.0     1.0    0.0     0.0   
2              ABT     55.035   5505200    0.0     0.0    0.0     0.0   
3    ACET_delisted      9.290     34700    0.0     0.0    1.0     1.0   
4             ACTG     15.890     70500    0.0     0.0    1.0     1.0   
..             ...        ...       ...    ...     ...    ...     ...   
512            XEL     22.030   1139800    0.0     0.0    0.0     0.0   
513            XOM     90.490  12833200    0.0     0.0    0.0     0.0   
514            XPO      1.710     39901    0.0     0.0    0.0     0.0   
515            ZBH     80.320    715129    0.0     0.0    1.0     1.0   
516           ZIXI      2.445   1263300    1.0     1.0    0.0     0.0   

         min_x  min_gp_dt      min_y  ... Debt_Per_Share_norm  \
0   2007-12-10 2007-12-10 2008-10-06  ...      

/tmp/ipykernel_19/2302199464.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  traindspn['Sector1']='dum'


In [ ]:
import datetime

from datetime import date




mm = [12]
mm = [1,2,3,4,5,6,7,8,9,10,11,12]
yy = [2026]

yy = [2018,2019,2020,2021,2022,2023,2024,2025,2026]

#yy = [2018]
#mm = [12]
dd=[12]
ct=0
for y in yy:
    for m in mm:
        for d in dd:
            dt1=datetime.date(y, m, d)
            
            if dt1>datetime.date(2026, 10, 13):
                continue
            #if dt1.weekday()>4:
            #    dt1=datetime.date(y, m, d+4)
            dt2=datetime.date(y+1, m, d)
            #if dt2.weekday()>4:
            #    dt2=datetime.date(y+1, m, d+4)
            dt0=datetime.date(y-1, m, d)
        
            print("dt0, dt1, dt2=", dt0, dt1, dt2)
            testds=get_current_state(dt1.strftime("%Y-%m-%d"), dt2.strftime("%Y-%m-%d"),dt0.strftime("%Y-%m-%d"),1.4,0.7)
            if testds.empty:
                print("testds empty")
                continue
            #testds=get_current_state(dt1, dt2,dt0,1.4,0.7)
            testds['Sector1']='dum'
            #print("testds=",testds)
            ds=doe(5, trainds,testds, 1.4,0.7,'logistic_reg_2023_02_40_30_5.csv',reg_o,reg_p,reg_on,reg_pn,testds['calc_open'].mean(), avg_x1o,avg_x2o,avg_x3o, avg_x4o,testds['calc_open'].mean(),avg_x1p,avg_x2p,avg_x3p, avg_x4p)
            ds['dt1']=dt1
            if ct>0:
                allds=allds.append(ds)
            else:
                allds=ds
                ct=1
print("allds=",allds)                

allds.to_csv('./allds3_oct13.csv')


#doe(5, './finds_03_2009_40_30.csv','./finds_02_2024_40_30.csv', 1.4,0.7,'logistic_reg_2023_02_40_30_5.csv')
#doe(5, './finds_07_2021_40_10.csv','./finds_07_2022_40_10.csv', 1.4,0.9,'logistic_2022_07_40_10_5.csv')
    #year1_df1a=pd.read_csv('./finds_07_2021.csv') 
    #year2_df1a=pd.read_csv('./finds_07_2022.csv')


In [17]:
print("allds=",allds)

allds=     exp_profit_rego  wt_profit_rego  exp_profit_rego1  wt_profit_rego1  \
0         -0.110818       -0.155723          2.150270         1.980017   
0         -0.029498       -0.155908          2.244424         1.800370   
0         -0.072840       -0.129961          2.269117         2.024944   
0         -0.054857       -0.121589          2.270909         2.010241   
0         -0.092513       -0.131599          2.213036         2.044948   
..              ...             ...               ...              ...   
0         -0.025057       -0.045041          4.406219         4.424211   
0         -0.057334       -0.147198          4.240509         3.841223   
0         -0.062262       -0.116919          4.290792         4.126393   
0         -0.047052       -0.126007          4.070550         3.777447   
0         -0.039976       -0.067868          4.164675         4.169031   

    exp_profit_regp  wt_profit_regp  exp_profit_regp1  wt_profit_regp1  \
0         -0.208035       -0.2